# Projeto Prático 3 — Predição de Séries Temporais com `TDNN` (`PMC` com Janela Deslizante)

> **Contexto de Engenharia de Controle e Produção (Seção 5.10 do Livro-Base):**
> Até agora nossos modelos tratavam amostras estáticas independentes. Mas e quando queremos prever o **próximo instante de um sistema dinâmico variante no tempo** — como vibração de turbina, demanda elétrica horária ou preço de uma *commodity* industrial?

Usamos uma **Rede Neural com Atraso no Tempo (*Time-Delay Neural Network* — `TDNN`)**: um **Perceptron Multicamadas** alimentado por uma **janela deslizante** das últimas $p$ leituras $[x(t-1), x(t-2), \dots, x(t-p)]$ para estimar o valor futuro $x(t)$!
- **Analogia de Engenharia de Controle:** Se em Sinais e Sistemas um filtro digital linear FIR calcula $\hat{x}(t) = \sum_{k=1}^{p} a_k x(t-k)$, a rede **TDNN** é um **filtro digital não linear autoajustável**!

---

## 1. Carregando o Histórico ($t = 1 \dots 100$) e o Horizonte Futuro ($t = 101 \dots 120$)

Seguindo fielmente a Seção 5.10 (Tabelas do Apêndice III e Tabela 5.7 do livro-base), avaliaremos **três janelas/topologias candidatas** com Backpropagation + *Momentum* ($\eta = 0{,}1$, $\alpha = 0{,}8$, $\varepsilon = 0{,}5 \times 10^{-6}$):
- **`TDNN 1`:** $p = 5$ atrasos $\rightarrow$ `10` neurônios ocultos $\rightarrow$ `1` saída (`5-10-1`)
- **`TDNN 2`:** $p = 10$ atrasos $\rightarrow$ `15` neurônios ocultos $\rightarrow$ `1` saída (`10-15-1`)
- **`TDNN 3`:** $p = 15$ atrasos $\rightarrow$ `25` neurônios ocultos $\rightarrow$ `1` saída (`15-25-1`)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Série histórica de treinamento (t = 1 até 100) e série futura real de teste (t = 101 até 120)
serie_treino_df = pd.read_csv("datasets/projeto-pratico-3.csv")
serie_teste_df = pd.read_csv("datasets/conjunto-testes-projeto-pratico-3.csv")

plt.figure(figsize=(11, 4))
plt.plot(serie_treino_df["t"], serie_treino_df["x"], color="royalblue", label="Histórico de Treino (t = 1..100)")
plt.plot(serie_teste_df["t"], serie_teste_df["x"], color="crimson", marker="o", markersize=4, label="Futuro de Validação (t = 101..120)")
plt.axvline(100.5, color="black", linestyle="--", label=" Fronteira Treino | Teste")
plt.title("Série Temporal Completa do Processo Dinâmico")
plt.xlabel("Instante Discreto de Tempo (t)")
plt.ylabel(" Grandeza Normalizada x(t)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

---

## 2. Construindo a Janela Deslizante (Transformando Série em Matriz Supervisionada)

Veja como uma janela de tamanho $p = 5$ transforma o vetor 1D $x(1 \dots 100)$ em pares $(\text{entradas}, \text{alvo})$:
- Amostra 1: entradas $[x(5), x(4), x(3), x(2), x(1)] \longrightarrow$ alvo desejado $x(6)$
- Amostra 2: entradas $[x(6), x(5), x(4), x(3), x(2)] \longrightarrow$ alvo desejado $x(7)$
- ... até $t = 100$ (gerando $100 - p$ padrões de treinamento!).

Para testar em $t = 101 \dots 120$ (predição **um passo à frente** conforme Tabela 5.7), o instante $t = 101$ olha imediatamente para as $p$ medidas anteriores $[x(100), x(99), \dots, x(101-p)]$!

In [ ]:
def montar_janelas_tdnn(serie_treino, serie_teste, p_atrasos):
    '''Constrói matrizes supervisionadas (X_tr, D_tr) e (X_te, D_te) para janela de tamanho p.'''
    # 1. Padrões de treinamento extraídos estritamente de t = 1..100 (sem qualquer vazamento do teste!)
    X_tr, D_tr = [], []
    for t_idx in range(p_atrasos, len(serie_treino)):
        # Janela [x(t-1), x(t-2), ..., x(t-p)] em ordem decrescente de atraso
        janela = serie_treino[t_idx - p_atrasos : t_idx][::-1]
        X_tr.append(janela)
        D_tr.append([serie_treino[t_idx]])

    # 2. Padrões de teste para t = 101..120 (usa os últimos p pontos conhecidos imediatamente antes de t)
    serie_completa = np.concatenate([serie_treino, serie_teste])
    X_te, D_te = [], []
    for t_idx in range(len(serie_treino), len(serie_completa)):
        janela = serie_completa[t_idx - p_atrasos : t_idx][::-1]
        X_te.append(janela)
        D_te.append(serie_completa[t_idx])

    return np.array(X_tr), np.array(D_tr), np.array(X_te), np.array(D_te)

# Verificando dimensões para p = 5
s_tr = serie_treino_df["x"].to_numpy()
s_te = serie_teste_df["x"].to_numpy()
X5_tr, D5_tr, X5_te, D5_te = montar_janelas_tdnn(s_tr, s_te, p_atrasos=5)
print(f"Para p=5 -> Treino X: {X5_tr.shape}, D: {D5_tr.shape} | Teste X: {X5_te.shape}, D: {D5_te.shape}")

---

## 3. Classe `TDNN` (PMC com *Momentum*) e Execução dos 9 Treinamentos (Tabela 5.6)

Executamos 3 treinamentos (`T1`, `T2`, `T3`) para cada uma das 3 topologias (`TDNN 1`, `TDNN 2` e `TDNN 3`) com $\eta = 0{,}1$, $\alpha = 0{,}8$ e $\varepsilon = 0{,}5 \times 10^{-6}$ (`max_epochs = 3000` para execução ágil em aula; sinta-se à vontade para testar `5000`!).

In [ ]:
class TDNN:
    def __init__(self, p_entradas, n_ocultos, seed=42):
        self.rng = np.random.default_rng(seed)
        self.W1 = self.rng.uniform(-0.5, 0.5, size=(n_ocultos, p_entradas + 1))
        self.W2 = self.rng.uniform(-0.5, 0.5, size=(1, n_ocultos + 1))

    @staticmethod
    def _sig(u):
        return 1.0 / (1.0 + np.exp(-np.clip(u, -50.0, 50.0)))

    def treinar(self, X, D, eta=0.1, alpha=0.8, epsilon=0.5e-6, max_epocas=3000):
        N = X.shape[0]
        hist_mse = []
        dW1_ant = np.zeros_like(self.W1)
        dW2_ant = np.zeros_like(self.W2)

        for epoca in range(1, max_epocas + 1):
            soma_e2 = 0.0
            for idx in self.rng.permutation(N):
                x_b = np.vstack([[-1.0], X[idx].reshape(-1, 1)])
                y1 = self._sig(self.W1 @ x_b)
                y1_b = np.vstack([[-1.0], y1])
                y2 = self._sig(self.W2 @ y1_b)

                e = D[idx].reshape(-1, 1) - y2
                soma_e2 += 0.5 * float(np.sum(e ** 2))

                delta2 = e * (y2 * (1.0 - y2))
                delta1 = (y1 * (1.0 - y1)) * (self.W2[:, 1:].T @ delta2)

                dW2 = eta * (delta2 @ y1_b.T) + alpha * dW2_ant
                self.W2 += dW2
                dW2_ant = dW2

                dW1 = eta * (delta1 @ x_b.T) + alpha * dW1_ant
                self.W1 += dW1
                dW1_ant = dW1

            mse = soma_e2 / N
            hist_mse.append(mse)
            if mse <= epsilon:
                return hist_mse, epoca
        return hist_mse, max_epocas

    def prever(self, X):
        preds = []
        for i in range(X.shape[0]):
            x_b = np.vstack([[-1.0], X[i].reshape(-1, 1)])
            y1_b = np.vstack([[-1.0], self._sig(self.W1 @ x_b)])
            preds.append(float(self._sig(self.W2 @ y1_b)[0, 0]))
        return np.array(preds)


topologias = [
    ("TDNN 1 (5-10-1)", 5, 10),
    ("TDNN 2 (10-15-1)", 10, 15),
    ("TDNN 3 (15-25-1)", 15, 25),
]

registros_tab56 = []
registros_tab57 = []
melhores_por_topologia = {}

for nome_top, p_in, n_hid in topologias:
    X_tr, D_tr, X_te, D_te = montar_janelas_tdnn(s_tr, s_te, p_atrasos=p_in)
    melhor_erro_rel = float("inf")

    for t_num, semente in enumerate([11, 22, 33], start=1):
        rede = TDNN(p_in, n_hid, seed=semente + p_in)
        hist, ep = rede.treinar(X_tr, D_tr, eta=0.1, alpha=0.8, epsilon=0.5e-6, max_epocas=2500)
        y_pred_te = rede.prever(X_te)

        # Erro relativo percentual em t = 101..120
        erros_rel = (np.abs(D_te - y_pred_te) / D_te) * 100.0
        media_rel = float(np.mean(erros_rel))
        var_rel = float(np.var(erros_rel))

        registros_tab56.append({
            "Topologia": nome_top, "Treino": f"T{t_num}",
            "MSE Final": hist[-1], "Épocas": ep
        })
        registros_tab57.append({
            "Topologia": nome_top, "Treino": f"T{t_num}",
            "Erro Relativo Médio (%)": media_rel, "Variância (%)": var_rel
        })

        if media_rel < melhor_erro_rel:
            melhor_erro_rel = media_rel
            melhores_por_topologia[nome_top] = {
                "treino": f"T{t_num}", "hist": hist, "y_pred": y_pred_te, "D_te": D_te, "erro_rel": media_rel
            }

print("Tabela 5.6 – Convergência no Treinamento:")
print(pd.DataFrame(registros_tab56).to_string(index=False))

print("\nTabela 5.7 – Desempenho na Predição Futura (t = 101..120):")
print(pd.DataFrame(registros_tab57).round(3).to_string(index=False))

---

## 4. Curvas de Convergência e Gráficos Real vs. Estimado ($t = 101 \dots 120$)

Seguindo os itens 4 e 5 da Seção 5.10, plotamos em páginas de **3 subgráficos não superpostos**:
1. A evolução do $E_M$ (`MSE`) do melhor modelo de cada topologia;
2. O acompanhamento temporal ponto a ponto entre o sinal real $x(t)$ e a previsão $\hat{x}(t)$ para $t = 101 \dots 120$.

In [ ]:
# 1. Três gráficos não superpostos do MSE por época
fig, axes = plt.subplots(3, 1, figsize=(9, 8))
for ax, (nome_top, info) in zip(axes, melhores_por_topologia.items()):
    ax.plot(info["hist"], color="teal", linewidth=1.8)
    ax.set_yscale("log")
    ax.set_title(f"Evolução do MSE — {nome_top} (Melhor: {info['treino']})")
    ax.set_xlabel("Época")
    ax.set_ylabel("MSE (log)")
    ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# 2. Três gráficos não superpostos de Valores Desejados vs. Estimados (t = 101..120)
instantes_teste = serie_teste_df["t"].to_numpy()
fig, axes = plt.subplots(3, 1, figsize=(10, 9))
for ax, (nome_top, info) in zip(axes, melhores_por_topologia.items()):
    ax.plot(instantes_teste, info["D_te"], "ko-", linewidth=1.8, label="Valor Real x(t)")
    ax.plot(instantes_teste, info["y_pred"], "r^--", linewidth=1.8, label=f"Previsto ({info['treino']})")
    ax.set_title(f"{nome_top} — Erro Relativo Médio: {info['erro_rel']:.2f}%")
    ax.set_xlabel("Instante de Tempo (t)")
    ax.set_ylabel("Amplitude")
    ax.set_xticks(instantes_teste)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**Lição Valiosa de Engenharia (Parcimônia vs. *Overfitting*):** Compare as três topologias! Aumentar demais a janela e o número de neurônios (`TDNN 3: 15-25-1` possui `426` pesos ajustáveis para menos de `85` janelas de treino!) nem sempre melhora a previsão futura — muitas vezes uma topologia mais enxuta (`TDNN 1` ou `TDNN 2`) generaliza melhor e consome muito menos memória embarcada!

---

## 5. Exercícios Práticos de Fixação

### Exercício 1: Contando o Número Total de Parâmetros (Pesos + Biases) de cada TDNN
Calcule e imprima quantos pesos ajustáveis totais existem em `TDNN 1 (5-10-1)`, `TDNN 2 (10-15-1)` e `TDNN 3 (15-25-1)`, lembrando que cada neurônio oculto tem `p + 1` pesos e o neurônio de saída tem `n1 + 1` pesos.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
for nome, p_in, n_hid in [("TDNN 1", 5, 10), ("TDNN 2", 10, 15), ("TDNN 3", 15, 25)]:
    total_pesos = n_hid * (p_in + 1) + 1 * (n_hid + 1)
    amostras_disponiveis = 100 - p_in
    print(f"{nome}: {total_pesos:3d} pesos ajustáveis para {amostras_disponiveis} janelas de treino")
```

</details>

### Exercício 2: Previsão Recursiva Multi-Passos (*Closed-Loop Forecasting*)
Na validação acima, em $t = 102$ alimentamos a rede com a leitura real $x(101)$ medida pelo sensor em tempo real. E se o sensor pifar em $t = 100$ e precisarmos prever `5` passos no futuro realimentando a **própria estimativa da rede** na janela? Implemente um loop de 5 passos recursivos usando o melhor modelo!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
# Treina rapidamente uma TDNN 1 (p=5) dedicada ao teste recursivo
X_tr5, D_tr5, _, _ = montar_janelas_tdnn(s_tr, s_te, p_atrasos=5)
modelo_rec = TDNN(5, 10, seed=11)
modelo_rec.treinar(X_tr5, D_tr5, max_epocas=1500)

# Começa com as últimas 5 leituras reais conhecidas [x(100), x(99), x(98), x(97), x(96)]
historico_vivo = list(s_tr[-5:])
previsoes_autonomas = []

for passo in range(5):
    janela_atual = np.array([historico_vivo[-5:][::-1]])
    prox_valor = modelo_rec.prever(janela_atual)[0]
    previsoes_autonomas.append(prox_valor)
    historico_vivo.append(prox_valor)  # Realimenta a própria previsão como se fosse medida!

for idx_p, (real_v, prev_v) in enumerate(zip(s_te[:5], previsoes_autonomas), start=101):
    print(f"t = {idx_p} -> Real: {real_v:.4f} | Previsão Autônoma Recursiva: {prev_v:.4f}")
```

</details>